In [1]:
import os
import json
from pathlib import Path

In [2]:
BASE_DIR = Path("./TextVQA")
NEW_DIR = Path("./TextVQA_tiny")
NEW_DIR.mkdir(exist_ok=True)

In [3]:
BASE_IMG_DIR = BASE_DIR / "train_images"
NEW_IMG_DIR = NEW_DIR / "images"
NEW_IMG_DIR.mkdir(exist_ok=True)

## 1. READ and FILTER the answer for each question

In [4]:
import random
from typing import List
from collections import Counter


def filter_answers(answers: List[str]) -> str:
    """
    Filter out answers that are most common in a list of answers
    :param answers: List of answers
    :return: Most common answer
    """
    frequency = Counter(answers)
    max_freq = max(frequency.values())
    tie_terms = [term for term, count in frequency.items() if count == max_freq]
    if len(tie_terms) == 0:
        return None
    if "unanswerable" in tie_terms:
        if len(tie_terms) > 1:
            tie_terms.remove("unanswerable")
            return tie_terms[0]
        else:
            return None
    else:
        random_term = random.choice(tie_terms)
        return random_term

In [5]:
target_fields = ["question", "image_id", "image_width", "image_height", "answers"]

with open(BASE_DIR / "TextVQA_0.5.1_train.json", "r") as f:
    train_set = json.load(f)["data"]
    train_set = [{k: v for k, v in d.items() if k in target_fields} for d in train_set]
    for d in train_set:
        d["answer"] = filter_answers(d["answers"])
    # drop unanswerable questions
    train_set = [d for d in train_set if d["answer"] is not None]
        

with open(BASE_DIR / "TextVQA_0.5.1_val.json", "r") as f:
    val_set = json.load(f)["data"]
    val_set = [{k: v for k, v in d.items() if k in target_fields} for d in val_set]
    for d in val_set:
        d["answer"] = filter_answers(d["answers"])
    # drop unanswerable questions
    val_set = [d for d in val_set if d["answer"] is not None]
        

len(train_set), len(val_set)

(34109, 4922)

In [6]:
random.seed(42)
random.shuffle(train_set)
random.shuffle(val_set)

train_set = train_set[:4000]
val_set = val_set[:1000]

In [7]:
for d in train_set:
    img_id = d["image_id"]
    os.system(f"cp {BASE_IMG_DIR}/{img_id}.jpg {NEW_IMG_DIR}/{img_id}.jpg")

for d in val_set:
    img_id = d["image_id"]
    os.system(f"cp {BASE_IMG_DIR}/{img_id}.jpg {NEW_IMG_DIR}/{img_id}.jpg")

In [8]:
with open(NEW_DIR / "train.json", "w") as f:
    json.dump(train_set, f, indent=4)

with open(NEW_DIR / "val.json", "w") as f:
    json.dump(val_set, f, indent=4)
    